# 📊 Data Analysis — 10-Week Course
## Week 7: Regression Analysis

---
### Learning Objectives
By the end of this week, you will be able to:
- Fit and interpret simple and multiple linear regression models
- Evaluate model fit (R², adjusted R², RMSE, residual plots)
- Build logistic regression models for binary outcomes
- Use regression for prediction and forecasting

### Outline
1. Regression Fundamentals
2. Simple Linear Regression
3. Multiple Linear Regression
4. Model Evaluation
5. Logistic Regression
6. Prediction & Forecasting
7. Lab Exercises
8. Assignment

---
## 1. Regression Fundamentals

### What Is Regression?
Regression models the **relationship between a dependent variable (Y) and one or more independent variables (X)**.

| Type | Y | X | Use |
|---|---|---|---|
| Simple Linear | Continuous | 1 numeric | Predict Y from one X |
| Multiple Linear | Continuous | ≥2 variables | Predict Y from several X |
| Logistic | Binary (0/1) | Any | Classify into two groups |
| Polynomial | Continuous | Curved | Non-linear relationships |

### Simple Linear Model
$$Y = \beta_0 + \beta_1 X + \varepsilon$$

| Term | Meaning |
|---|---|
| β₀ | Intercept — predicted Y when X = 0 |
| β₁ | Slope — change in Y per unit increase in X |
| ε | Residual error |

### OLS Assumptions (LINE)
- **L**inearity — relationship between X and Y is linear
- **I**ndependence — observations are independent
- **N**ormality — residuals are normally distributed
- **E**qual variance (homoscedasticity) — residuals have constant variance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                              accuracy_score, classification_report,
                              confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

import os
if os.path.exists("africa_health_cleaned.csv"):
    df = pd.read_csv("africa_health_cleaned.csv")
else:
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })

# Log-transform skewed predictors
df["log_gdp"]           = np.log(df["gdp_per_capita"])
df["log_infant_mort"]   = np.log(df["infant_mortality"])
df["log_maternal_mort"] = np.log(df["maternal_mortality"])
print("Dataset ready:", df.shape)

---
## 2. Simple Linear Regression

We predict `life_expectancy` from `log_gdp`.

In [ ]:
# ── Method 1: scipy.stats.linregress ──────────────────────────────────
x = df["log_gdp"]
y = df["life_expectancy"]

slope, intercept, r, p, se = stats.linregress(x, y)

print("Simple Linear Regression: LE ~ log(GDP)")
print(f"  Intercept (β₀) = {intercept:.4f}")
print(f"  Slope     (β₁) = {slope:.4f}")
print(f"  r (Pearson)    = {r:.4f}")
print(f"  R²             = {r**2:.4f}")
print(f"  p-value        = {p:.6f}")
print(f"  Std Error (β₁) = {se:.4f}")
print()
print(f"  Equation: LE = {intercept:.2f} + {slope:.2f} × log(GDP)")
print(f"  Interpretation: A 1-unit increase in log(GDP) → "
      f"{slope:.2f} years more life expectancy")

In [ ]:
# Regression plot with prediction band
x_fit  = np.linspace(x.min(), x.max(), 200)
y_fit  = slope * x_fit + intercept

# 95% prediction interval (approximate)
n    = len(x)
x_mean = x.mean()
residuals = y - (slope * x + intercept)
s_res = np.sqrt(np.sum(residuals**2) / (n - 2))
se_fit = s_res * np.sqrt(1/n + (x_fit - x_mean)**2 / np.sum((x - x_mean)**2))
t_crit = stats.t.ppf(0.975, df=n-2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: regression line + PI
ax = axes[0]
PALETTE = {
    "West Africa": "#3498DB", "East Africa": "#2ECC71",
    "North Africa": "#E74C3C", "Central Africa": "#9B59B6",
    "Southern Africa": "#F39C12",
}
for region, grp in df.groupby("region"):
    ax.scatter(grp["log_gdp"], grp["life_expectancy"],
               color=PALETTE[region], alpha=0.75, s=65,
               edgecolors="white", label=region)

ax.plot(x_fit, y_fit, "k-", lw=2, label="Regression line")
ax.fill_between(x_fit,
                y_fit - t_crit * se_fit,
                y_fit + t_crit * se_fit,
                color="gray", alpha=0.2, label="95% PI")

ax.text(0.05, 0.95,
        f"LE = {intercept:.1f} + {slope:.2f}·log(GDP)\nR² = {r**2:.3f}",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax.set_xlabel("Log GDP per Capita")
ax.set_ylabel("Life Expectancy (yrs)")
ax.set_title("A — Simple Linear Regression", fontweight="bold")
ax.legend(fontsize=7)

# Panel B: Residual plot
ax = axes[1]
y_pred = slope * x + intercept
resid  = y - y_pred
ax.scatter(y_pred, resid, color="#3498DB", alpha=0.7, edgecolors="white", s=60)
ax.axhline(0, color="red", ls="--", lw=1.5)
ax.set_xlabel("Fitted Values")
ax.set_ylabel("Residuals")
ax.set_title("B — Residual Plot", fontweight="bold")

plt.suptitle("Simple Linear Regression — LE ~ log(GDP)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("week7_slr.png", bbox_inches="tight")
plt.show()

---
## 3. Multiple Linear Regression

$$LE = \beta_0 + \beta_1 \cdot \log(GDP) + \beta_2 \cdot Vaccination + \beta_3 \cdot WaterAccess + \beta_4 \cdot HealthExpend + \varepsilon$$

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

features = ["log_gdp", "vaccination_dtp3", "water_access", "health_expenditure"]
X = df[features]
y = df["life_expectancy"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

mlr = LinearRegression()
mlr.fit(X_train, y_train)

y_pred_train = mlr.predict(X_train)
y_pred_test  = mlr.predict(X_test)

r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

# Adjusted R² for training set
n_train, k = len(X_train), len(features)
adj_r2 = 1 - (1 - r2_train) * (n_train - 1) / (n_train - k - 1)

print("Multiple Linear Regression — Life Expectancy")
print("-" * 50)
print(f"  Intercept: {mlr.intercept_:.4f}")
for feat, coef in zip(features, mlr.coef_):
    print(f"  {feat:<25} β = {coef:+.4f}")
print()
print(f"  Train R²       = {r2_train:.4f}")
print(f"  Adj. R²        = {adj_r2:.4f}")
print(f"  Test  R²       = {r2_test:.4f}")
print(f"  Test  RMSE     = {rmse_test:.4f} years")

In [ ]:
# Visualise MLR results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A — Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred_test, color="#3498DB", alpha=0.8,
           edgecolors="white", s=70)
lims = [min(y_test.min(), y_pred_test.min()) - 1,
        max(y_test.max(), y_pred_test.max()) + 1]
ax.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax.set_xlabel("Actual Life Expectancy")
ax.set_ylabel("Predicted Life Expectancy")
ax.set_title(f"A — Actual vs Predicted (R²={r2_test:.3f})", fontweight="bold")
ax.legend()

# B — Coefficient plot
ax = axes[1]
coef_df = pd.DataFrame({"Feature": features, "Coefficient": mlr.coef_})
coef_df = coef_df.sort_values("Coefficient")
colors_c = ["#E74C3C" if c < 0 else "#2ECC71" for c in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"],
        color=colors_c, edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coefficient Value")
ax.set_title("B — Regression Coefficients", fontweight="bold")

# C — Residuals distribution
ax = axes[2]
residuals_test = y_test.values - y_pred_test
ax.hist(residuals_test, bins=12, color="#9B59B6", edgecolor="white", alpha=0.85)

# Normal overlay
xr = np.linspace(residuals_test.min(), residuals_test.max(), 100)
normal_curve = stats.norm.pdf(xr, residuals_test.mean(), residuals_test.std())
ax2 = ax.twinx()
ax2.plot(xr, normal_curve, "r--", lw=2, label="Normal curve")
ax2.set_yticks([])
ax2.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)

ax.set_xlabel("Residual (actual − predicted)")
ax.set_ylabel("Count")
ax.set_title("C — Residual Distribution", fontweight="bold")

plt.suptitle("Multiple Linear Regression Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week7_mlr.png", bbox_inches="tight")
plt.show()

---
## 4. Model Evaluation

### Regression Metrics
| Metric | Formula | Interpretation |
|---|---|---|
| **R²** | 1 − SS_res/SS_tot | Proportion of variance explained (0–1) |
| **Adjusted R²** | Penalises for extra predictors | Use when comparing models with different # features |
| **RMSE** | √mean(residuals²) | Average prediction error (same units as Y) |
| **MAE** | mean(|residuals|) | Less sensitive to outliers than RMSE |

### Residual Diagnostics
| Plot | What to Check |
|---|---|
| Residuals vs Fitted | Random scatter → linearity OK; funnel shape → heteroscedasticity |
| Q-Q plot of residuals | Points on diagonal → normality OK |
| Scale-Location | Horizontal band → equal variance |
| Residuals vs Leverage | Cook's distance → influential points |

In [ ]:
# Full diagnostic plots for the MLR model (train set)
y_pred_all = mlr.predict(X)
resid_all  = y.values - y_pred_all

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# A — Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(y_pred_all, resid_all, color="#3498DB", alpha=0.7,
           edgecolors="white", s=55)
ax.axhline(0, color="red", ls="--", lw=1.5)
ax.set_xlabel("Fitted Values")
ax.set_ylabel("Residuals")
ax.set_title("A — Residuals vs Fitted", fontweight="bold")

# B — Q-Q Plot
ax = axes[0, 1]
(quantiles, values), (slope_qq, intercept_qq, r_qq) = stats.probplot(resid_all)
ax.scatter(quantiles, values, color="#3498DB", alpha=0.7, s=55)
ax.plot(quantiles, slope_qq * np.array(quantiles) + intercept_qq,
        "r--", lw=1.5)
ax.set_xlabel("Theoretical Quantiles")
ax.set_ylabel("Sample Quantiles")
ax.set_title("B — Q-Q Plot of Residuals", fontweight="bold")

# C — Scale-Location
ax = axes[1, 0]
sqrt_abs_resid = np.sqrt(np.abs(resid_all))
ax.scatter(y_pred_all, sqrt_abs_resid, color="#2ECC71", alpha=0.7,
           edgecolors="white", s=55)
# LOWESS smoother (manual)
from scipy.ndimage import uniform_filter1d
sort_idx = np.argsort(y_pred_all)
smoothed = uniform_filter1d(sqrt_abs_resid[sort_idx], size=8)
ax.plot(y_pred_all[sort_idx], smoothed, "r-", lw=1.5)
ax.set_xlabel("Fitted Values")
ax.set_ylabel("√|Residuals|")
ax.set_title("C — Scale-Location", fontweight="bold")

# D — Residual histogram
ax = axes[1, 1]
ax.hist(resid_all, bins=15, color="#9B59B6", edgecolor="white", alpha=0.85, density=True)
xr = np.linspace(resid_all.min(), resid_all.max(), 100)
ax.plot(xr, stats.norm.pdf(xr, resid_all.mean(), resid_all.std()),
        "r--", lw=2, label="Normal")
ax.set_xlabel("Residuals")
ax.set_ylabel("Density")
ax.set_title("D — Residual Distribution", fontweight="bold")
ax.legend()

fig.suptitle("Regression Diagnostic Plots", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week7_diagnostics.png", bbox_inches="tight")
plt.show()

---
## 5. Logistic Regression

Logistic regression models the **probability of a binary outcome**.

$$P(Y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \ldots)}}$$

- Output: probability between 0 and 1
- Decision boundary: P ≥ 0.5 → class 1
- Coefficients are in **log-odds** units; exponentiate for **odds ratios**

In [ ]:
# Binary target: high_le (life expectancy ≥ median)
le_median = df["life_expectancy"].median()
df["high_le"] = (df["life_expectancy"] >= le_median).astype(int)

log_features = ["log_gdp", "vaccination_dtp3", "water_access",
                "health_expenditure", "log_infant_mort"]
X_log = df[log_features]
y_log = df["high_le"]

# Scale features
scaler = StandardScaler()
X_log_s = scaler.fit_transform(X_log)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_log_s, y_log, test_size=0.25, random_state=42, stratify=y_log
)

log_reg = LogisticRegression(random_state=42, max_iter=500)
log_reg.fit(X_tr, y_tr)

y_pred_lr = log_reg.predict(X_te)
y_prob_lr = log_reg.predict_proba(X_te)[:, 1]

acc = accuracy_score(y_te, y_pred_lr)

print(f"Logistic Regression — Predict High Life Expectancy")
print(f"  Accuracy = {acc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_te, y_pred_lr,
                             target_names=["Low LE", "High LE"]))

# Odds ratios
print("\nOdds Ratios (exp(β)):")
for feat, coef in zip(log_features, log_reg.coef_[0]):
    print(f"  {feat:<25} OR = {np.exp(coef):.4f}")

In [ ]:
# Confusion matrix + ROC curve
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# A — Confusion matrix
ax = axes[0]
cm = confusion_matrix(y_te, y_pred_lr)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Low", "Predicted High"],
            yticklabels=["Actual Low",    "Actual High"],
            ax=ax)
ax.set_title(f"A — Confusion Matrix (Acc={acc:.2f})", fontweight="bold")

# B — ROC Curve
ax = axes[1]
fpr, tpr, _ = roc_curve(y_te, y_prob_lr)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="#3498DB", lw=2, label=f"ROC AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.fill_between(fpr, tpr, alpha=0.1, color="#3498DB")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("B — ROC Curve", fontweight="bold")
ax.legend()

plt.suptitle("Logistic Regression — Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week7_logistic.png", bbox_inches="tight")
plt.show()

---
## 6. Prediction & Forecasting

In [ ]:
# Predict life expectancy for hypothetical new countries
new_countries = pd.DataFrame({
    "log_gdp"           : [np.log(500), np.log(2000), np.log(8000)],
    "vaccination_dtp3"  : [45, 72, 90],
    "water_access"      : [40, 65, 88],
    "health_expenditure": [2.0, 4.5, 7.0],
})
new_countries["predicted_le"] = mlr.predict(new_countries)

# Add labels
new_countries["scenario"] = ["Low income", "Middle income", "High income"]

print("Predicted Life Expectancy for Hypothetical Countries:")
print(new_countries[["scenario", "log_gdp", "vaccination_dtp3",
                      "water_access", "predicted_le"]].to_string(index=False))

# Visualise predictions
fig, ax = plt.subplots(figsize=(8, 4))
colors_scenario = ["#E74C3C", "#F39C12", "#2ECC71"]
bars = ax.barh(new_countries["scenario"], new_countries["predicted_le"],
               color=colors_scenario, edgecolor="white", alpha=0.9)
for bar, val in zip(bars, new_countries["predicted_le"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f} yrs", va="center", fontsize=10, fontweight="bold")
ax.set_xlabel("Predicted Life Expectancy (years)")
ax.set_title("MLR Predictions: Three Economic Scenarios", fontweight="bold")
ax.axvline(df["life_expectancy"].mean(), color="gray", ls="--",
           label=f"Observed mean {df['life_expectancy'].mean():.1f}")
ax.legend()
plt.tight_layout()
plt.savefig("week7_predictions.png", bbox_inches="tight")
plt.show()

---
## 7. Lab Exercises

### Lab 1: Simple Regression
Fit a simple linear regression to predict `infant_mortality` from `vaccination_dtp3`.  
Print the equation, R², and RMSE. Plot regression line + residuals.

In [ ]:
# Lab 1
# TODO: SLR — infant_mortality ~ vaccination_dtp3
pass

### Lab 2: Model Comparison
Build two MLR models to predict `life_expectancy`:
- Model A: only `log_gdp`
- Model B: `log_gdp` + `vaccination_dtp3` + `water_access`

Compare train/test R² and RMSE. Which model is better, and why?

In [ ]:
# Lab 2
# TODO: Build Model A and Model B, compare metrics

### Lab 3: Logistic Regression for Vaccination
Create a binary target `high_vax` (vaccination_dtp3 ≥ 80).  
Fit a logistic regression using `log_gdp`, `water_access`, and `health_expenditure`.  
Report accuracy, classification report, and plot the confusion matrix.

In [ ]:
# Lab 3
df["high_vax"] = (df["vaccination_dtp3"] >= 80).astype(int)
# TODO: Logistic regression for high_vax

---
## 8. Assignment — Week 7

**Problem 1 (25 pts):** Build a multiple linear regression to predict `maternal_mortality`.  
Use at least 4 predictors (justify your choice).  
Report: equation, coefficients, R², RMSE, and full diagnostic plots.

**Problem 2 (25 pts):** Implement polynomial regression (degree 2) for `life_expectancy ~ water_access`.  
Compare R² of the linear vs polynomial model.  
Plot both curves on the same scatter plot.

**Problem 3 (25 pts):** Build and evaluate a logistic regression to predict whether a country has `hiv_prevalence > 5%`.  
Include: confusion matrix, classification report, ROC curve, and odds ratios.

**Problem 4 (25 pts):** Write a `regression_report(X, y, feature_names)` function that:
- Fits a LinearRegression model
- Prints the equation, R², Adj R², RMSE
- Returns a DataFrame of coefficients with magnitude-based ranking

In [ ]:
# Problem 1
pass  # TODO

In [ ]:
# Problem 2 — Polynomial regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
# TODO

In [ ]:
# Problem 3 — Logistic regression for HIV > 5%
df["high_hiv"] = (df["hiv_prevalence"] > 5).astype(int)
# TODO

In [ ]:
# Problem 4
def regression_report(X, y, feature_names):
    # TODO
    pass

---
## Summary — Week 7

| Concept | Key Point |
|---|---|
| Simple linear regression | Y = β₀ + β₁X; slope = change in Y per unit X |
| Multiple linear regression | Add predictors to explain more variance |
| OLS assumptions | Linearity, Independence, Normality, Equal variance (LINE) |
| R² | Proportion of variance explained |
| Adjusted R² | Penalises for adding uninformative predictors |
| RMSE | Average prediction error in Y units |
| Diagnostic plots | Check residuals for patterns (non-linearity, heteroscedasticity) |
| Logistic regression | Binary outcome; coefficients are log-odds |
| Odds ratio | exp(β) — multiplicative change in odds |

**Next:** Week 8 — Classification Algorithms